In [7]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [8]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country, ud.country_code as country_code,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year in (2023,2022)
    AND ud.month = 12
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region, ud.country_code




""")
# Convert query results into dataframe
df = regional_uniques.copy()

df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

# Initialize an empty DataFrame to hold all results
final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country_code', 'wmf_region'], 
                                       columns='month', 
                                       values='total_unique_devices',
                                       aggfunc='sum').reset_index()
    #Merge
    country_code_name_mapping = df[['country_code', 'country']].drop_duplicates(subset='country_code')

    # Merge the pivot table with the country name from the country_code_name_mapping DataFrame
    df_pivot = pd.merge(df_pivot, country_code_name_mapping, on='country_code', how='left')
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                         'formula_result_percentage_regional', 'simple_calculation_regional', 
                         'rank_simple_calculation', 'rank_formula_result',
                         'simple_calculation_global', 'formula_result_percentage_global']]
    final_df.to_csv("unique_devices_time_series_analysis_mobile.csv", index=False)

else:
    print("Saving CSV file")
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                          'simple_calculation_regional', 
                         'simple_calculation_global','formula_result_percentage_regional', 'formula_result_percentage_global']]
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy.csv", index = False, header=False)
    
# Show df
final_df

Saving CSV file


,month,country,wmf_region,current_metric,previous_metric,absolute_change,simple_calculation_regional,simple_calculation_global,formula_result_percentage_regional,formula_result_percentage_global
134,2023-12,Russia,Central & Eastern Europe & Central Asia,66836326.0,70205648.0,3369322.0,26.662810,2.430973,3.824936,4.462146
131,2023-12,Poland,Central & Eastern Europe & Central Asia,22228151.0,23945391.0,1717240.0,13.589216,1.238992,12.040319,2.061713
139,2023-12,Turkey,Central & Eastern Europe & Central Asia,25385598.0,27039845.0,1654247.0,13.090727,1.193543,10.120161,2.088153
115,2023-12,Bulgaria,Central & Eastern Europe & Central Asia,5241488.0,4009805.0,1231683.0,9.746807,0.888662,16.594801,0.920045
141,2023-12,Uzbekistan,Central & Eastern Europe & Central Asia,4797389.0,3816502.0,980887.0,7.762156,0.707712,13.420991,0.715699
...,...,...,...,...,...,...,...,...,...,...
186,2023-12,São Tomé and Príncipe,Sub-Saharan Africa,6062.0,5956.0,106.0,0.001319,0.000076,0.043499,0.000077
155,2023-12,Western Sahara,Sub-Saharan Africa,121.0,151.0,30.0,0.000373,0.000022,0.000038,0.000029
189,2023-12,French Southern Territories,Sub-Saharan Africa,2.0,18.0,16.0,0.000199,0.000012,0.000400,0.000014
154,2023-12,Djibouti,Sub-Saharan Africa,41315.0,41308.0,7.0,0.000087,0.000005,0.277706,0.001145


## Mobile enwiki breakdown

In [9]:
# Query

regional_uniques_mobile = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country, ud.country_code as country_code,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_domain_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.year in (2023,2022)
    AND ud.month = 12
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region, ud.country_code




""")

# Convert query results into dataframe
df = regional_uniques_mobile.copy()

df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country_code', 'wmf_region'], 
                                       columns='month', 
                                       values='total_unique_devices',
                                       aggfunc='sum').reset_index()
    #Merge
    country_code_name_mapping = df[['country_code', 'country']].drop_duplicates(subset='country_code')

    # Merge the pivot table with the country name from the country_code_name_mapping DataFrame
    df_pivot = pd.merge(df_pivot, country_code_name_mapping, on='country_code', how='left')
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                         'formula_result_percentage_regional', 'simple_calculation_regional', 
                         'rank_simple_calculation', 'rank_formula_result',
                         'simple_calculation_global', 'formula_result_percentage_global']]
    final_df.to_csv("unique_devices_time_series_analysis_mobile.csv", index=False)

else:
    print("Saving CSV file")
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                          'simple_calculation_regional', 
                         'formula_result_percentage_regional', 'simple_calculation_global', 'formula_result_percentage_global']]
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy_mobile.csv", index = False, header = False)
    
# Show df
final_df

Saving CSV file


,month,country,wmf_region,current_metric,previous_metric,absolute_change,simple_calculation_regional,formula_result_percentage_regional,simple_calculation_global,formula_result_percentage_global
141,2023-12,Uzbekistan,Central & Eastern Europe & Central Asia,8045236.0,5541807.0,2503429.0,24.493361,20.615668,1.510234,1.645802
140,2023-12,Ukraine,Central & Eastern Europe & Central Asia,29888911.0,28620922.0,1267989.0,12.405909,5.458684,0.764935,0.039373
115,2023-12,Bulgaria,Central & Eastern Europe & Central Asia,7323796.0,6209467.0,1114329.0,10.902511,8.648237,0.672237,0.619030
125,2023-12,Kazakhstan,Central & Eastern Europe & Central Asia,12446944.0,11622429.0,824515.0,8.066993,5.184544,0.497402,0.242564
131,2023-12,Poland,Central & Eastern Europe & Central Asia,33866930.0,34665221.0,798291.0,7.810419,10.039945,0.481582,1.600321
...,...,...,...,...,...,...,...,...,...,...
162,2023-12,Equatorial Guinea,Sub-Saharan Africa,40812.0,41216.0,404.0,0.013450,0.054399,0.000244,0.001549
155,2023-12,Western Sahara,Sub-Saharan Africa,144.0,187.0,43.0,0.001432,0.001368,0.000026,0.000037
186,2023-12,São Tomé and Príncipe,Sub-Saharan Africa,7770.0,7729.0,41.0,0.001365,0.014786,0.000025,0.000205
189,2023-12,French Southern Territories,Sub-Saharan Africa,3.0,32.0,29.0,0.000965,0.001083,0.000017,0.000022
